# CV Deep Learning - Tugas Deep Learning

**Proyek Computer Vision untuk Analisis Rambut Salon (TIEN SALON)**

**Catatan Penting:**
- Notebook ini mengkonsolidasi preprocessing, training experiments, dan evaluation pipeline CV deep learning untuk klasifikasi jenis dan panjang rambut.
- **Tidak termasuk RAG evaluation** (RAG eval dipisahkan jika diperlukan).
- Output visualization disimpan di `apps/api/cv/outputs/`

---

## Daftar Isi

1. [Section 1: CONFIGURATION & IMPORTS](#section-1-config--imports)
2. [Section 2: PREPROCESSING PIPELINE](#section-2-preprocessing-pipeline)
3. [Section 3: TRAINING EXPERIMENTS](#section-3-training-experiments)
4. [Section 4: EVALUATION](#section-4-evaluation)

### Section 1: CONFIGURATION & IMPORTS

### Penjelasan

Bagian ini mengatur semua konfigurasi awal untuk eksperimen deep learning:
- **Seeds**: Menetapkan seed untuk reproducibility (random number generators dari PyTorch dan Python)
- **ImageNet normalization**: Normalisasi menggunakan mean=[0.485,0.456,0.406] dan std=[0.229,0.224,0.225] sesuai ImageNet statistics
- **Paths**: Path ke dataset mentah dan folder output untuk artifacts
- **Imports**: Semua library yang diperlukan (PyTorch, torchvision, PIL, dll.)

Configurasi ini penting karena memastikan eksperimen dapat direproduksi dengan hasil yang sama setiap kali dijalankan.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps
import random
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Configuration
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
INPUT_SIZE = 224
RESIZE_SIDE = 256
CROP = 224
MIN_SIDE = 224

# Augmentation for training
RRC_SCALE = (0.875, 1.0)
RRC_RATIO = (3.0 / 4.0, 4.0 / 3.0)
FLIP_P = 0.5
ROT_DEG = 10
AUG_REPEAT = 2

# Training hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Paths
DATASET = Path('dataset/figaro1k')
OUT = Path('outputs')

### Section 2: PREPROCESSING PIPELINE

### Penjelasan

Pipeline preprocessing terdiri dari:
- **RGB 8-bit sRGB**: Input harus RGB, bukan grayscale (warna penting untuk salon)
- **EXIF transpose**: Auto-orient sesuai EXIF orientation
- **Resize keep-aspect**: Resize sisi pendek ke 256px
- **Center crop 224**: Crop center 224x224
- **ToTensor + Normalize**: Convert ke Tensor + ImageNet normalization
- **Output**: Tensor NCHW float32 (1,3,224,224)

**Augmentasi (training only):**
- RandomResizedCrop(scale 0.875–1.0, ratio 3/4–4/3)
- RandomHorizontalFlip(p=0.5)
- RandomRotation(degrees=±10°)
- Setiap citra train → (1+2) = 3 versi
- Val/inference identik tanpa augmentasi

In [ ]:
def _train_transforms() -> transforms.Compose:
    """Transforms untuk training dengan augmentasi."""
    return transforms.Compose([
        transforms.RandomResizedCrop(
            CROP,
            scale=RRC_SCALE,
            ratio=RRC_RATIO,
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        ),
        transforms.RandomHorizontalFlip(p=FLIP_P),
        transforms.RandomRotation(degrees=ROT_DEG, interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def _val_transforms() -> transforms.Compose:
    """Transforms untuk validation/inference (tanpa augmentasi)."""
    return transforms.Compose([
        transforms.Resize(RESIZE_SIDE, interpolation=InterpolationMode.BILINEAR, antialias=True),
        transforms.CenterCrop(CROP),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def _to_nchw(img: Image.Image, basic: transforms.Compose) -> torch.Tensor:
    """Convert PIL Image ke NCHW tensor."""
    pipe = transforms.Compose([
        basic,
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return pipe(img).unsqueeze(0)   # (1,3,224,224)


def _load_rgb(path: Path) -> Image.Image | None:
    """Load RGB image, skip if invalid."""
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im)
            if im.mode != "RGB":
                return None
            if min(im.size) < MIN_SIDE:
                return None
            return im.convert("RGB")
    except Exception:
        return None

# Example usage
CLASSES = ["lurus", "bergelombang", "keriting", "sangat-keriting"]
CLASS_INDICES = {cls: i for i, cls in enumerate(CLASSES)}

print("Train transforms created successfully")
print("Val transforms created successfully")
print("Class indices:", CLASS_INDICES)
print("Device:", DEVICE)

### Section 3: TRAINING EXPERIMENTS

### Penjelasan

Bagian ini mencakup berbagai eksperimen training backbone dan ablation studies:
- **Backbone benchmark**: Benchmark 15+ backbone architectures untuk classification task
- **Ablation study**: Variasi seeds (s1, s7, s123) untuk melihat stabilitas
- **Ensemble strategies**: Combination of multiple checkpoints dengan weighted averaging
- **Best model selection**: Memilih model terbaik berdasarkan accuracy/F1 score
- **Export**: Export best checkpoint ke ONNX format

Contoh backbone yang di-benchmark:
- MobileNetV2
- EfficientNet-B0/B1/B2
- ResNet-18/34/50
- ConvNeXt-Tiny
- Swin-Tiny
- etc.

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Define model backbones to benchmark
BACKBONE_MODELS = {
    'mobilenet_v2': models.mobilenet_v2(pretrained=True),
    'efficientnet_b0': models.efficientnet_b0(pretrained=True),
    'resnet18': models.resnet18(pretrained=True),
    'convnext_tiny': models.convnext_tiny(pretrained=True),
}

# Modify head for hair classification (4 classes)
def modify_head(model, num_classes=4):
    if 'mobilenet' in model.name:
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif 'efficientnet' in model.name:
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif 'resnet' in model.name:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif 'convnext' in model.name:
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

# Training loop example
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')
    avg_loss = running_loss / len(dataloader)
    
    return avg_loss, acc, f1

# Example usage: Initialize model
model_name = 'mobilenet_v2'
model = modify_head(getattr(models, model_name)(pretrained=True))
model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
print(f"Model {model_name} initialized and moved to {DEVICE}")

### Section 4: EVALUATION

### Penjelasan

Bagian ini mencakup evaluasi komprehensif terhadap model terbaik:
- **Confusion matrices**: Multi-class confusion matrix heatmaps untuk hair type & hair length
- **Per-class metrics**: Bar charts untuk precision/recall/F1 per kelas
- **Separability analysis**: Boxplot F-statistics dan scatter plots untuk keriting vs sangat-keriting
- **Comparison tables**: Tabel perbandingan dengan geometric labeling dan Gemini methods
- **Summary report**: Export summary report ke JSON

Visualizations yang dihasilkan:
- Confusion matrix heatmap (hair type & hair length)
- Bar chart precision/recall/F1 per kelas
- Separability scatter plot (keriting vs sangat-keriting)
- Comparison table vs other methods

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
import pandas as pd

# Function to generate confusion matrix
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', fmt='.0f',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{OUT}/confusion_matrix_{title.replace(" ", "_").lower()}.png', dpi=300)
    plt.close()
    print(f"Confusion matrix saved to {OUT}/confusion_matrix_{title.replace(' ', '_').lower()}.png")

# Example: Generate confusion matrix for test predictions
# Example test predictions (replace with actual model predictions)
y_true_test = [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
y_pred_test = [0, 1, 2, 3, 0, 1, 3, 2, 0, 1, 2, 3, 0, 1, 3, 2, 0, 1, 2, 3]

# Generate confusion matrix
plot_confusion_matrix(y_true_test, y_pred_test, CLASSES, 'Hair-Type-Classification')

# Generate classification report
report = classification_report(y_true_test, y_pred_test, target_names=CLASSES, output_dict=True)
print(classification_report(y_true_test, y_pred_test, target_names=CLASSES))

# Save classification report
df_report = pd.DataFrame(report).transpose()
df_report.to_csv(f'{OUT}/classification_report.csv')
print(f"Classification report saved to {OUT}/classification_report.csv")